# Ma Soi — Behavior Cloning on Colab (P0/P1)

Trains the same model family as `village-bc-0002` (the one in production): AdamW +
cosine + early stopping, SiLU + LayerNorm, separate value trunk (`masoi-mlp-2`).
Same flags as section 3 of `train_bc_local.ipynb`, so the numbers are comparable.

**BC only.** The RL loop (section 6 of the local notebook) stays local: rollouts are
TypeScript self-play on CPU, and free Colab has 2 vCPU — slower than your machine.
A GPU only speeds up the PPO step, which is seconds.

## Step A — on your machine: package the dataset

The encoded dataset `dataset-0004` is already in `.tmp/enc` (1,066,738 rows). Package it
together with the training code **from the same checkout** (the zip carries
`masoi_training/`, so Colab trains with exactly your code):

```powershell
Compress-Archive -Path ai-training\masoi_training, .tmp\enc -DestinationPath .tmp\bc-package.zip -Force
```

~2.9 GB raw; the zip is smaller but still large. To cut ~0.9 GB, drop
`scores.f32.bin` first (only used with `--distill-alpha > 0`, default 0) — see
"Out of RAM" at the end.

Need a NEW dataset instead? Generate, validate and encode it first
(`--no-jitter` is required, the validator must print `dataset SẠCH`):

```powershell
npx tsx apps/server/scripts/selfplay.ts --games 10000 --players 8 --preset --defense --seed bc --trajectories .tmp/bc --trace-games 10000 --no-jitter --quiet
npm run ai:validate-dataset -- .tmp/bc/trajectories.jsonl
npm run ai:encode -- --in .tmp/bc/trajectories.jsonl --out .tmp/enc
```

## Step B — Drive

Upload `bc-package.zip` to Google Drive, folder **`MyDrive/masoi/`** (or change
`DRIVE_DIR` in section 1).

## Step C — Colab

1. Open this notebook in Colab (File → Upload notebook, or open it from Drive/GitHub).
2. **Runtime → Change runtime type → T4 GPU.**
3. Run the cells top to bottom. Every cell stops with a clear message if something is
   wrong — read that message instead of re-running blindly.
4. Section 6 copies the model to `MyDrive/masoi/models/<run>/`. Download that folder and
   follow the "back on your machine" steps there.

Boundary (§39): game rules + encoder run on your machine in TypeScript. Colab only sees numbers.

## 0. Check GPU

Colab sometimes gives you a CPU runtime silently. No GPU: **Runtime → Change runtime type → T4 GPU**.
Training still works on CPU, just slower.

In [ ]:
# torch moved the ONNX exporter to a separate package (local runs report
# "No module named 'onnxscript'"). Install here: train_bc runs in a subprocess,
# so it takes effect without a restart. ONNX is optional - the game reads JSON.
%pip install -q onnxscript

import torch

print("torch", torch.__version__, "| cuda build", torch.version.cuda)
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU  {name}  ({vram:.1f} GB)")
else:
    print("NO GPU - Runtime > Change runtime type > T4 GPU, then re-run this cell.")

## 1. Load data from Drive

In [ ]:
import os
import time
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/masoi"       # where bc-package.zip lives
ZIP = f"{DRIVE_DIR}/bc-package.zip"
RUN = time.strftime("bc-p0-%Y%m%d-%H%M")        # output folder name, also on Drive
OUT = f"/content/{RUN}"

if not os.path.exists(ZIP):
    raise SystemExit(f"{ZIP} not found - upload bc-package.zip to Drive (Step B) or fix DRIVE_DIR")

!rm -rf /content/bc && mkdir -p /content/bc
!unzip -q "$ZIP" -d /content/bc
!ls /content/bc /content/bc/enc
print("run:", RUN)

## 2. Validate data before training

Three questions before spending GPU time:

1. Do files match `meta.json` — a truncated file still reshapes "fine" and shifts
   every label by a row.
2. Do train/val/test sizes sum to the full set (§15)?
3. **Is every label LEGAL under its own mask** — otherwise the dataset teaches
   out-of-rule moves?

In [ ]:
import gc
import glob
import os
import sys

sys.path.insert(0, "/content/bc")
from masoi_training.data import action_distribution, load

# Auto-find the dataset dir: the full package names it `enc`, the slim one `enc-slim`.
found = [p for p in sorted(glob.glob("/content/bc/*")) if os.path.isfile(os.path.join(p, "meta.json"))]
if len(found) != 1:
    raise SystemExit(f"need exactly 1 dir with meta.json under /content/bc, saw: {found}")
DATA = found[0]
print("dataset dir:", DATA)


def check(path):
    """Check inside a function so numpy arrays die on return.

    `load()` puts ALL tensors in RAM (~3.1 GB for the full package). If `data`
    lived at notebook scope, the train cell's subprocess would load ANOTHER copy
    on top of its own train/val/test splits - past Colab free RAM, killed with
    exit -9 (SIGKILL) and an empty log. Seen it happen. Return numbers, not arrays.
    """
    data = load(path)  # throws if a file mismatches meta.json

    sizes = {name: len(data.split(name)) for name in ("train", "validation", "test")}
    assert sum(sizes.values()) == len(data)
    assert data.masks[range(len(data)), data.actions].all(), "labels point at ILLEGAL actions"
    assert set(data.rewards.tolist()) <= {-1.0, 1.0}

    # `optimal.u8.bin` is what `agreementTieAware` relies on. Without it the §17
    # headline metric is null and only `agreement` (which penalizes tied moves) remains.
    assert data.optimal is not None, "missing optimal.u8.bin - re-run ai:encode"

    return {
        "version": data.meta.get("datasetVersion"),
        "commit": (data.meta.get("gitCommit") or "")[:8],
        "rows": len(data),
        "obs": data.obs_size,
        "actions": data.action_size,
        "games": data.meta.get("games"),
        "sizes": sizes,
        "classes": len(action_distribution(data)),
        "scores": data.scores is not None,
    }


info = check(DATA)
gc.collect()

print("dataset  ", info["version"], "commit", info["commit"])
print("rows     ", info["rows"], "| obs", info["obs"], "| actions", info["actions"])
print("games    ", info["games"])
print("split    ", info["sizes"])
print("action classes (§43):", info["classes"])
print("scores   ", "present (only used with --distill-alpha > 0)" if info["scores"] else "absent - not needed by default")
print("\nOK - ready to train. RAM was released before the train cell.")

## 3. Smoke test: 2 epochs

Catches config errors in about a minute instead of after the full run. Uses the SAME
flags as the full train, so a flag the packaged code doesn't know fails here.

In [ ]:
import subprocess
import sys

# Same config as village-bc-0002 (train_bc_local.ipynb section 3). `masoi-mlp-2` needs
# silu/layernorm/separate: the game engine reads that format.
FLAGS = [
    "--batch-size", "512",
    "--lr", "1e-3",
    "--hidden", "128",
    "--seed", "12345",
    "--optimizer", "adamw", "--weight-decay", "0.01",
    "--scheduler", "cosine", "--warmup-epochs", "1",
    "--grad-clip", "1.0", "--patience", "5",
    "--init", "orthogonal",
    "--activation", "silu",
    "--norm", "layernorm",
    "--value-trunk", "separate", "--value-weight", "0.5",
]


def run_logged(cmd, log_path):
    # Stream the child's output into this cell AND a log file. Returns the exit code.
    start = time.time()
    with open(log_path, "w", encoding="utf8") as log, subprocess.Popen(
        cmd, cwd="/content/bc", stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf8", errors="replace",
    ) as proc:
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
    print(f"\ntotal {time.time() - start:.0f}s | exit {proc.returncode}")
    return proc.returncode


code = run_logged(
    [sys.executable, "-m", "masoi_training.train_bc", "--data", DATA, "--out", "/content/smoke",
     "--epochs", "2", "--model-id", "smoke", *FLAGS],
    "/content/smoke.log",
)
if code != 0:
    raise SystemExit(f"smoke test failed (exit {code}) - read the log above")
print("smoke OK")

## 4. Full train

Up to 100 epochs; `--patience 5` stops early and keeps the best epoch (local run:
best epoch 37, 554 s on CPU). The net is small, so a T4 is bounded by per-step
overhead — don't expect a big speed-up over a decent CPU.

Colab can disconnect (idle ~90 min, hard limit ~12 h). `train_bc` writes its output
only at the end, so keep the tab open until `exit 0`. The log is also copied to Drive.

In [ ]:
MODEL_ID = "policy-p0"   # packaging renames it (e.g. village-bc-0003); keep it generic here

!rm -rf "$OUT"
code = run_logged(
    [sys.executable, "-m", "masoi_training.train_bc", "--data", DATA, "--out", OUT,
     "--epochs", "100", "--model-id", MODEL_ID, *FLAGS],
    f"/content/{RUN}.log",
)
!mkdir -p "$DRIVE_DIR/logs" && cp "/content/$RUN.log" "$DRIVE_DIR/logs/"
if code != 0:
    raise SystemExit(
        f"train_bc failed (exit {code}). Read the log above.\n"
        "exit -9 = killed for RAM, log often empty: see 'Out of RAM' at the end."
    )

## 5. Read the results

The deciding number is **`metrics.test.agreementTieAware`** — does the model pick a
TIED-FOR-BEST move vs the teacher on unseen games? Dataset ceiling: **0.984**.
Reference, `village-bc-0002`: test tieAware **0.9723**, agreement 0.9282.

Not `agreement`: it penalizes tied moves the teacher broke by raw id — info §9 hides
from the observation, so no model can learn it. And don't judge by loss.

In [ ]:
import json
import pathlib

path = pathlib.Path(OUT) / "metrics.json"
if not path.exists():
    raise SystemExit(f"no {path} - the train cell did NOT finish. Read its last 'exit ...' line.")
report = json.loads(path.read_text())
m = report["metrics"]

print(f"{'':<11} {'tieAware':>9} {'agreement':>10} {'top-2':>8}")
for name in ("train", "validation", "test"):
    e = m[name]
    print(f"{name:<11} {e.get('agreementTieAware'):>9} {e.get('agreement'):>10} {e.get('top2Agreement'):>8}")
print(f"\nvs village-bc-0002: test tieAware {m['test']['agreementTieAware'] - 0.9723:+.4f}, "
      f"agreement {m['test']['agreement'] - 0.9282:+.4f}")

gap = m["train"]["agreementTieAware"] - m["validation"]["agreementTieAware"]
print(f"train - val = {gap:+.4f}  ->", "OVERFIT" if gap > 0.05 else "no overfit")
print("best epoch:", report["bestEpoch"], "/", report["trainingConfig"]["epochs"])

print("\nBy decision type (worst first):")
for kind, score in sorted(m["test"].get("agreementByDecision", {}).items(), key=lambda kv: kv[1]):
    print(f"  {kind:<13} {score}")
print("\nBy role (worst first):")
for role, score in sorted(m["test"].get("agreementByRole", {}).items(), key=lambda kv: kv[1]):
    print(f"  {role:<18} {score}")
print("\nCalibration (ECE, > 0.1 is overconfident): test", m["test"].get("ece"))

w = json.loads((pathlib.Path(OUT) / "model.weights.json").read_text())
print("\nexport:", w["format"], w.get("activation"), w.get("norm"), w.get("valueTrunk"),
      "| onnx:", "ok" if not report.get("onnxError") else report["onnxError"])

In [ ]:
import matplotlib.pyplot as plt

history = report["history"]
epochs = [row["epoch"] for row in history]

figure, left = plt.subplots(figsize=(8, 4))
left.plot(epochs, [row["trainLoss"] for row in history], label="train loss")
left.set_xlabel("epoch")
left.set_ylabel("loss")

right = left.twinx()
right.plot(
    epochs,
    [r["val_agreementTieAware"] if r.get("val_agreementTieAware") is not None else r.get("val_agreement") for r in history],
    color="tab:orange",
    label="val agreement",
)
right.set_ylabel("agreement")

figure.legend(loc="upper right")
plt.title("Behavior cloning: falling loss is NOT enough, agreement must rise")
plt.show()

## 6. Save the model to Drive

Copies `model.weights.json` (what the game reads), `metrics.json` (modelId, gitCommit,
datasetVersion, seed — §46) and `model.pt` to `MyDrive/masoi/models/<run>/`.

### Back on your machine

1. Download the folder from Drive into `.tmp/<run>/` (e.g. `.tmp/bc-p0-20260918-1030/`).
2. Benchmark it against production — same seeds as the local comparison:

```bash
npm run ai:benchmark -- --model .tmp/<run>/model.weights.json --setups baseline,village,wolves,all,teacher --learned-decisions vote,night,final,hunter
```

3. Send the benchmark summary and section 5 output to Claude. Packaging into
   `apps/server/assets/models/` (new file name, tests, env) is done from there.

NEVER overwrite `apps/server/assets/models/village-bc-0002.weights.json` — production
and rollback both depend on it.

In [ ]:
DEST = f"{DRIVE_DIR}/models/{RUN}"
!mkdir -p "$DEST"
!cp "$OUT/model.weights.json" "$OUT/metrics.json" "$OUT/model.pt" "$DEST/"
!cp "$OUT/model.onnx" "$DEST/" 2>/dev/null || true
!ls -lh "$DEST"

# Or download directly (a few MB):
# from google.colab import files
# !cd /content && zip -qr "$RUN.zip" "$RUN"
# files.download(f"/content/{RUN}.zip")

## Reading the numbers

Read `test.agreementTieAware`, against the **dataset ceiling**, not 1.0.

The ceiling is the "agreement ceiling (§17)" line `npm run ai:validate-dataset` prints.
For the 10,002-game dataset of 2026-09-16 it is **0.984**. The rest is moves the teacher
broke with info absent from the observation; no policy can learn that part.

| tieAware / ceiling | Meaning | Do |
|---|---|---|
| < 0.35 | Barely learned anything | See below |
| 0.35 – 0.70 | Trend learned, bot not replicated | Raise `--epochs` / `--hidden` |
| > 0.75 | Baseline replicated well | §17 cleared, RL is on the table |

### Agreement stalled?

Look at `agreementByDecision` **before** touching the model. `SPEECH` has no labels in
this action space — 862,075 / 1,928,813 rows are "unlabeled" for exactly that reason —
so if one decision type drags the average, that's where to look, not `--hidden`.

Raise `--hidden` only when train ≈ val and both are low (underfit). If train ≫ val,
a bigger model feeds the disease.

Don't seed-shop. Changing `--seed` tests stability; if two seeds disagree widely,
neither number concludes anything.

## Out of RAM (exit -9)

`exit -9` is SIGKILL: Linux killed the process for RAM. The log is usually EMPTY,
so it never names the cause.

`masoi_training.data.load()` loads ALL tensors into RAM, then `split()` copies three
more. On the 10,002-game dataset:

| | full package | slim package |
|---|---|---|
| full | 3.08 GB | 2.21 GB |
| + train / val / test | 2.12 + 0.48 + 0.48 | 1.52 + 0.34 + 0.34 |
| **one process** | **6.16 GB** | **4.41 GB** |

Colab free has ~12.7 GB. One process fits. **Two** don't — and that's the trap: if the
check cell keeps `data` alive, the notebook holds 3.08 GB while the train subprocess
loads 6.16 GB more. The check cell above already returns RAM right after checking.

Still OOM? Drop `scores.f32.bin` — only used with `--distill-alpha > 0` (default 0):

```powershell
Remove-Item .tmp\enc\scores.f32.bin
Compress-Archive -Path ai-training\masoi_training, .tmp\enc -DestinationPath .tmp\bc-package.zip -Force
```

`data.py` treats the file as optional, nothing else to change.

Measured (full dataset, process RSS): 0.03 GB before check → 3.12 GB inside (5.25 GB peak
while splitting) → 0.03 GB after return. RAM fully reclaimed before train runs.